---
title: Reflections on Minnowbrook
---

I had a nice time at the Minnowbrook analytic reasoning seminar.

Gorgeous lake, good food, nice chats, logic programming. What more is there to life?

Those black flies are no joke though. Now that I'm mostly healed up it is an amusing memory, but during the hike I looked back to blood smeared all on the back of my arms due to those little fuckers. I had a bite on the back off my neck that got so big and juicy I could feel the skin of my face being pulled back. I could _hear_ it.


For me, where my mind is at today, the conference was a little datalog heavy, and also perhaps a little too datalog _implementation_ heavy. I am always adjusting my self identity and preferences and open to change, but I do see themes in what drives me. Performance isn't really it. Simplicity, expressivity, and elegance are. Now these unfortunately are more subjective, so one person's solution may not be another's.

To some degree what I enjoy about datalog is how impoverished it is, that it is a mental challenge to write basic programs. In this respect I like datalog akin to how another person might like brainfuck or programming a minecraft computer or someting. Datalog is missing features that you didn't even know you could miss in a programming language because bottom up execution gives you no ordinary control flow. From this pain I think a deeper understanding comes, just as I think formalization of logic or implementation of an algorithm leads to deeper understanding. There is something to push on. 

This pathetic weakness of ordinary datalog is part of why I latch on to every datalog extension I hear about because the language is so awful. The unlock from each new feature is quite a bit higher. My personal definition of datalog (and yes I reserve the right to use terms in nonstandard ways. Fuck off.) is at minimum bottom up logic programming with functions symbols. I don't consider bottom up logic programming with only atoms an interesting enough language to play with, but it's good to know it's there.

The thing that is truly beautiful about datalog to me is that I think it a computable microcosm of what _logic is_. When I say logic, I mean the structure that exists even before propositional logic, before you introduce symbols like `and` `or` `implies` and the like. Logic starts with base axioms (judgements) and builds new judgements using rules (horizontal inference lines). It is impossible to give a satisfying definition to "logic", but defining it as the study of inductively defined sets is at least something to push on. https://arxiv.org/abs/2304.13430 The Logic of Logic Programming.

I also like datalog because I view prolog style unification variables with suspicion and kind of think they are somehow a broken mistake. One thing I'd like to try is make a prolog with unification removed (only pattern matching) and explicit moding (kind of multiple dispatch based on the current instantiation level). I do kind of wonder if Robinson gave us the apple of sin. Ground is godly, variables are sinful. This feeling is somewhat informed on my time playing with datalog and egraphs, both of which emphasize ground terms, and by from my opinion the failure of regular prolog to have a story for manipulating expressions with binding forms.

This is one reason I find it compelling 



If I were a better me, I would have tried to put together a talk on what are interesting datalog programs. https://www.philipzucker.com/notes/Languages/datalog/


It was also interesting to get a new sampling 





# Bits and Bobbles

Simple python datalog

What is datalog for?

https://inst.eecs.berkeley.edu/~cs294-260/sp24/2024-02-05-datalog
https://www.philipzucker.com/notes/Languages/datalog/
https://www.philipzucker.com/datalog-book


http://www.mmrc.iss.ac.cn/~xgao/paper/jar-gdbase.pdf deducative databases
gappa
the chase
lambda datalog
hypothetical datalog
first class sets


LDBC https://github.com/ldbc
graph500

The benchmarks

https://www.uni-muenster.de/imperia/md/content/informatik/agmueller-olm/publications/icalp04.pdf karr's algorotihm for discovering linear realtions


In [3]:
edge = {(1,2), (2,3), (3,1)}
path  = set()

while True:
    le, lp = len(edge), len(path)
    path |= edge
    path |= {(x,z) for x,y in edge 
                   for y1,z in path 
                   if y == y1}
    if le == len(edge) and lp == len(path):
        break
path

{(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (3, 1), (3, 2), (3, 3)}

Ugly, but the datalog program wasn't like a paragon of beauty either. Poor pattern matching and synatctic verboseit of for,if,in all kind of hurt.

In [17]:
from pprint import pprint
ev = set()
terms = {("+", ((2, ()) , (3, ())))}

while True:
    lt, le = len(terms), len(ev)
    terms |= {arg for (f, args) in terms for arg in args}
    ev |= { ((f,args), f)  for (f, args) in terms if isinstance(f, int)}
    #ev |= { ((f,args), va + vb)  for (f, args) in terms if f == "+" 
    #                       for (a, va) in ev 
    #                       if a == args[0]
    #                       for (b, vb) in ev 
    #                       if b == args[1] }
    ev |= { (("+",(a,b)), va + vb) 
                           for (a, va) in ev
                           for (b, vb) in ev
                           if ("+", (a,b)) in terms}
    if lt == len(terms) and le == len(ev):
        break
terms
pprint(ev)

{((2, ()), 2), ((3, ()), 3), (('+', ((2, ()), (3, ()))), 5)}


In [ ]:
from dataclasses import dataclass, field
from typing import Callable, TypeVar

class MergeDict[A,B]():
    merge : Callable[[A,A], A]
    data : dict[A,B] = field(default_factory=dict)
    def __or__(self, other):
        return MergeDict({**self, **other})
    

# union, lattice, semigroup, semiring, take old, programmable


Using multiple dispatch and differing types to implement multimodeness.
or subscripts
using generators for non determinism. Just writing the loops or a yield from

def edge_bf(x : int, y : RefCell)

Or _all_ values are wrapped in ref cells and you should check
aliasing mutable slice might be useful.
Ordered vs unordered comma


In [22]:
type Ref = list
def edge_bf(x : int, y : Ref):
    match x:
        case 1:
            y[0] = 2; yield
            y[0] = 3; yield
        case 2:
            y[0] = 3; yield

def edge_bb(x : int, y : int):
    if (x,y) in {
        (1,2),
        (1,3),
        (2,3)
    }:
        yield
# could generate combinator out of 
def from_rel(data):
    def res(*args):
        # lookup empty ref cells

res = [None]
for _ in edge_bf(1,res):
    print(res[0])


2
3


timely dataflow and dynamic stratification


In [ ]:
# https://docs.python.org/3/library/re.html#writing-a-tokenizer
from typing import NamedTuple
import re

class Token(NamedTuple):
    type: str
    value: int | float | str
    line: int
    column: int

def tokenize(code):
    token_spec = [
        ('WS',   r'\s*'),
        ("DOT", r"\."),
        ("IMPLIES", r":-"),
        ('NUMBER',   r'\d+(\.\d*)?'), 
        ('ID',       r'[A-Za-z_]\w*'),  # Identifiers
        ("LPAREN", r"\("),
        ("RPAREN", r"\)"),
        ('COMMA', r','),            # Comma
        ('MISMATCH', r'.'),            # Any other character
    ]
    tok_regex = '|'.join('(?P<%s>%s)' % pair for pair in token_spec)
    line_num = 1
    line_start = 0
    for mo in re.finditer(tok_regex, code):
        kind = mo.lastgroup
        value = mo.group()
        column = mo.start() - line_start
        if kind == 'NUMBER':
            value = float(value) if '.' in value else int(value)
        elif kind == 'ID':
            kind = value
        elif kind == 'NEWLINE':
            line_start = mo.end()
            line_num += 1
            continue
        elif kind == 'SKIP':
            continue
        elif kind == 'MISMATCH':
            raise RuntimeError(f'{value!r} unexpected on line {line_num}')
        yield Token(kind, value, line_num, column)





Murderbot on the road. Nice drive.

arrived at 6
opeing
good din
chatted with mochael

slog
ascent - 
gpu datalog
srdatalog. worst case optimal gpu datalog. llvm of datalog


logic programming
datalog with extensions

souffle adts aren't well indexed

scheme's macro system sucks - don't design for sugar. design for big dsls

redfox
mpi
cuda

gdlog

Edweg. typed multi lyer ir. incA
MLIR based datalog engine
https://dl.acm.org/doi/10.1145/3689767


yapall
cclyzer
abstrating abstrsct machines
langston

HISA
hash indexed sorted array


integeating lattices


datalog as a database of a single relation with the lattice being the powerset?

Relational union find + datalog?
DMT
first class sets

incrementsal
differential
zset? https://www.feldera.com/blog/z-sets-representing-database-changes

overflow vram


differential union find +- a union.


Egglog supports sets. Why am I not using it for inverse method?

# XSB prolog
David S Warren
motaqgue grammar PTQ
ATN grammar system
CWL with maier - "datalog"

Beeri tutorial MS as tabling POD 89
magic sets - how to push selection through joins n recursion

OLDT

CORAL Ramakrishnan


assign monce
multi machine model



unix fork based prolog?


memoizing functional vs reation languages

naurally handles stratified negatoin. "completion" we know we've done it all.

magic set applied to hornsat. proplog

modes importatnt for tabling

xsb for transitive clousre, faster than clingo and souffle
Efficiency of analysis of trasntiie relation. https://arxiv.org/abs/2504.21291

features
variant and subsumptive tabling
indexing
aggregation
modeling state
metaprogramming
parametrized moudle system


subsumptive vs variant
higher overhead, needs efficient retrieval of subsets

indexing
trie indexing

extensional relations. User specified before rows are loaded
star indexing

multiple indices can be decalred tables


aggregation
pairtal-order keeps only minimal
lattice 
abelian subgroup
programmable join



non stratified negation


well founded seamntics
3 valued
polynoimoal propsotional complexity (quadratic)

stable model semantics 
2 values
NP-hard postiojanl complexity

MOdeling state
tries are canonical
models

metaprogramming
asbtratc interpretation
model checking
explainers, debugger
rule/2


parametrized modules 






https://github.com/SWI-Prolog/tabled-prolog-book
https://arxiv.org/pdf/1012.5123 XSB: Extending Prolog with Tabled Logic Programming


^ existential some kind of gensym to get bag semantics in tabling?

# Federico Donato. OWL

typed intuitionsitc logic fro ontology

owl for assertions and sparql for queries

Tbox and Abox

:freewilly rdf:type bio:Whale

kripke model

want open world not closed world



# prela remy



# frank
datatoad

https://github.com/frankmcsherry/datatoad/tree/main/presentation

columnar wcoj


facts()
count()
join()

ExecAtom

PlanAtom
terms -> Set<T>
modes -> (lower, option(upper))

iterator bound


columnar extend environment

rels
envs = []
for v in vars:
    extenders = sorted([ rel for rel in rels if v in rel.terms()], key=lambda rel: rel.count)
    driver = extenders[0] # smallest extender
    newenvs = []
    for env in envs:
        for env1 in driven.extend(env, v):
            if all(rel1.accept(env1), for rel1 in extenders[1:]):
                newenvs.append(env1)
    envs = 


https://github.com/frankmcsherry/blog/blob/master/posts/2025-12-03.md

https://www.vldb.org/pvldb/vol11/p691-ammar.pdf
    



columnar is visiting data kind of breadthwise rather than in a depth first search way
factorized databases
indices. You can know ahead of time which orders you'll want.
Resorting of "assignments" env can be fruitful. Especiall for incremental


In [25]:
from dataclasses import dataclass, field
@dataclass
class Var():
    name : str

#class Rel(Protocol):
#    def count(self, )
"""
class Eq(Rel):
    lhs : Var
    rhs : Var
"""

@dataclass
class DataRel():
    data : set
    vars : tuple[Var | None]

    def count(self, env : list[object]) -> int:
        return len(self.extend(env))
    def extend(self, env : tuple[object]) -> set[tuple[object]]:
        assert isinstance(env, tuple)
        n = len(env)
        j = self.vars.index(n) # index to extend from
        return {env + (d[j],) for d in self.data if all(x == env[v] for x,v in zip(d, self.vars) if v < n)}
    def accept(self, env : list[object]) -> bool:
        n = len(env)
        for d in self.data:
            if all(x == env[v] for x,v in zip(d, self.vars) if v < n):
                return True
        return False


R = DataRel({(1,2), (1,5), (2,3)}, [0,1])
assert R.extend((1,)) == {(1,2), (1,5)}
assert R.accept((1,2))
assert not R.accept((1,3))
assert R.count((1,)) == 2







In [8]:
(1,2,3).index(2)

1

# Hedgelog




# gpu

von neumann one program counter, scalar registers

lose ooo branch predication units canm squeeze more useful stuff in tere


simt vs simd

simt allows addressing. Built in address.

read the whiteppaper for the chip before you even start
calculate guess of speed before you code


# flowlog

yannakakid 81
semijoin
fast I guess on differential datalow

Using the frontier for deletion
Deletion is sort of key
stratification deletes old strate
.output directives


magic set as demand for prolog modes
magic set as prolog 
XSB inverse method



differential dataflow


What is datalog for?
Program analysis?

demand transformation

dominator tree was hard for datalog



Proving that nonmonotonic stuff is sound




Give a term ordering to atoms
The program _is_ stratifiable





# annie liu

even reach 2*n steps

double win game
barber paradox
class count
https://arxiv.org/pdf/1606.06269

A semantics generalizing? well founded semantics / stable set?

close

https://par.nsf.gov/servlets/purl/10594827 Alga

https://arxiv.org/abs/2007.13053 semantics for aggregation



henglein easy join


In [ ]:
import scryer
m = scryer.Machine()
m.load_module_string("", """
%:- use_module(library(tabling)).
%:- table path/2.
edge(1,2).
edge(2,3).
edge(3,1).

path(X,Y) :- edge(X,Y).
path(X,Y) :- edge(X,Z), path(Z,Y).
""")
m.query("path(X,Y).")

[{'X': Integer { value: 1 }, 'Y': Integer { value: 1 }},
 {'Y': Integer { value: 2 }, 'X': Integer { value: 1 }},
 {'Y': Integer { value: 3 }, 'X': Integer { value: 1 }},
 {'X': Integer { value: 2 }, 'Y': Integer { value: 1 }},
 {'Y': Integer { value: 2 }, 'X': Integer { value: 2 }},
 {'X': Integer { value: 2 }, 'Y': Integer { value: 3 }},
 {'Y': Integer { value: 1 }, 'X': Integer { value: 3 }},
 {'X': Integer { value: 3 }, 'Y': Integer { value: 2 }},
 {'X': Integer { value: 3 }, 'Y': Integer { value: 3 }}]

# Binary Joins Hangdong Zhao

https://vldb.org/2026/
https://pages.cs.wisc.edu/~hangdong/


yannakakis not practical. input + output
https://en.wikipedia.org/wiki/Yannakakis_algorithm

bitmap push down in sql server


umbra

TPC5
https://www.tpc.org/information/benchmarks5.asp


skew


pull vs push

pandas



# Yihong
Vecotrizing Linear Recurrences

prefix sum
 f(x) = f(x-1) + g(x)

low pass
f(x) = 1.8*f(x-1) - 0.9 * f(x-2) + g(x)

Distance trasnform https://en.wikipedia.org/wiki/Distance_transform
Needleman  Wunsch https://en.wikipedia.org/wiki/Needleman%E2%80%93Wunsch_algorithm

F = A * F + G . 
convolution over semiring


You can mush arround recurrences

turn f(x-1) into 

f(x-2) + f(x-4) + ...

Rec(A,G) = mu F. F = AF + G
https://gcc.gnu.org/pipermail/gcc/2005-July/147635.html

chains of recurrences

https://mediatum.ub.tum.de/doc/1273784/85954.pdf Algebraic Systems of Fixpoint Equations
over Semirings: Theory and Applications





https://github.com/cmuparlay/parlaylib

# ballantyne
multi stage relational prorammign

interpreter in synthesizer can synthesize deep language


specialize and gather

Is grather like bagofall?

# denis ctadl
jadx

flowy
sarif
https://github.com/sandialabs/ctadl
ascent

Kris group has ascent based decompiler (50kloc) what.
